# 🦷 Diş Röntgeni Anomali Tespiti — Model Eğitim Notebook'u

Bu notebook, projede kullanılan YOLOv8 tabanlı diş röntgeni modelinin **eğitim sürecini belgelemek ve gerektiğinde modeli yeniden eğitmek** için hazırlanmıştır.

Bu dosya özellikle GitHub portföy sunumu için iki nedenle projede tutulur:

1. Modelin hangi veri yapısıyla ve hangi eğitim akışıyla üretildiğini göstermek,
2. Ham veri seti tekrar eklendiğinde eğitim sürecini yeniden çalıştırılabilir hale getirmek.

> **Not:** Günlük model testi, ekran görüntüsü ve LinkedIn demosu için `02_model_demo.ipynb` dosyasını kullanın. Bu notebook eğitim/reprodüksiyon amaçlıdır.

> ⚠️ **Klinik Uyarı:** Bu proje araştırma ve portföy prototipidir. Model çıktıları klinik tanı, tedavi planı veya hekim değerlendirmesinin yerine geçmez.

## 1. Ortamı ve Proje Yollarını Hazırlama

Bu bölümde proje kök dizini bulunur, veri seti yolu tanımlanır ve eğitim çıktılarının nereye kaydedileceği belirlenir.

Beklenen ham veri seti yapısı:

```text
data/
├── train/
│   ├── _annotations.csv
│   └── görüntüler
├── valid/
│   ├── _annotations.csv
│   └── görüntüler
└── test/
    ├── _annotations.csv
    └── görüntüler
```

In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil
from typing import Iterable

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import Markdown, display
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
DATA_YAML_PATH = DATA_DIR / "dental_data.yaml"
RUNS_DIR = PROJECT_ROOT / "runs" / "detect"
RUN_NAME = "dental_yolov8n"
TRAINED_WEIGHTS_PATH = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
PORTFOLIO_MODEL_PATH = PROJECT_ROOT / "models" / "best.pt"
SPLITS = ("train", "valid", "test")
SUPPORTED_IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp")

print("✅ Proje yolları hazırlandı.")
print(f"Proje kökü: {PROJECT_ROOT}")
print(f"Veri klasörü: {DATA_DIR}")
print(f"Eğitim çıktısı: {RUNS_DIR / RUN_NAME}")

## 2. Veri Setini Kontrol Etme

Bu hücre, her bölümde `_annotations.csv` dosyası ve görüntü dosyaları olup olmadığını kontrol eder. Ham veri seti GitHub reposuna eklenmediği için yeniden eğitim yapmak isteyen kullanıcıların veri setini yerel olarak `data/` klasörüne yerleştirmesi gerekir.

In [ ]:
def check_dataset_structure(data_dir: Path = DATA_DIR, splits: Iterable[str] = SPLITS) -> bool:
    """YOLO eğitiminden önce beklenen klasör ve CSV yapısını kontrol eder."""
    all_ok = True

    for split in splits:
        split_dir = data_dir / split
        annotation_file = split_dir / "_annotations.csv"
        image_count = 0

        if split_dir.exists():
            image_count = sum(
                1 for path in split_dir.iterdir()
                if path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
            )

        status = "✅" if split_dir.exists() and annotation_file.exists() and image_count > 0 else "⚠️"
        print(f"{status} {split:<5} | klasör: {split_dir.exists()} | CSV: {annotation_file.exists()} | görsel: {image_count}")

        if not split_dir.exists() or not annotation_file.exists() or image_count == 0:
            all_ok = False

    return all_ok

DATASET_READY = check_dataset_structure()

if not DATASET_READY:
    display(Markdown(
        "**Not:** Ham veri seti bulunamadı veya eksik. Eğitim hücrelerini çalıştırmadan önce "
        "`data/train`, `data/valid` ve `data/test` klasörlerini README'deki yapıya göre hazırlayın."
    ))

## 3. CSV Etiketlerini YOLO Formatına Dönüştürme

Veri setindeki `_annotations.csv` dosyaları Pascal VOC tarzı koordinatlar içerir:

```text
xmin, ymin, xmax, ymax
```

YOLO eğitimi için bu değerler aşağıdaki normalize formata dönüştürülür:

```text
class_id x_center y_center width height
```

Bu sürüm, notebook tekrar çalıştırıldığında aynı `.txt` dosyasına etiketlerin tekrar tekrar eklenmesini önlemek için label dosyalarını kontrollü biçimde yeniden yazar.

In [ ]:
REQUIRED_COLUMNS = {"filename", "width", "height", "class", "xmin", "ymin", "xmax", "ymax"}


def load_annotation_csv(csv_path: Path) -> pd.DataFrame:
    """CSV dosyasını okur ve zorunlu kolonları doğrular."""
    df = pd.read_csv(csv_path)
    missing_columns = REQUIRED_COLUMNS.difference(df.columns)
    if missing_columns:
        raise ValueError(f"{csv_path} içinde eksik kolonlar var: {sorted(missing_columns)}")
    return df


def collect_class_names(data_dir: Path = DATA_DIR, splits: Iterable[str] = SPLITS) -> list[str]:
    """Tüm split'lerdeki sınıfları ilk görülme sırasına göre toplar."""
    class_names: list[str] = []

    for split in splits:
        csv_path = data_dir / split / "_annotations.csv"
        if not csv_path.exists():
            continue

        df = load_annotation_csv(csv_path)
        for class_name in df["class"].dropna().astype(str):
            if class_name not in class_names:
                class_names.append(class_name)

    return class_names


def convert_csv_to_yolo_labels(data_dir: Path = DATA_DIR, splits: Iterable[str] = SPLITS) -> list[str]:
    """CSV anotasyonlarını YOLO `.txt` etiket dosyalarına dönüştürür."""
    class_names = collect_class_names(data_dir, splits)
    if not class_names:
        raise FileNotFoundError("Sınıf listesi oluşturulamadı. `_annotations.csv` dosyalarını kontrol edin.")

    class_to_id = {class_name: idx for idx, class_name in enumerate(class_names)}

    for split in splits:
        split_dir = data_dir / split
        csv_path = split_dir / "_annotations.csv"
        if not csv_path.exists():
            print(f"⚠️ {csv_path} bulunamadı, atlandı.")
            continue

        df = load_annotation_csv(csv_path)

        for image_name, group in df.groupby("filename", sort=False):
            label_lines: list[str] = []

            for _, row in group.iterrows():
                image_width = float(row["width"])
                image_height = float(row["height"])
                xmin, ymin, xmax, ymax = map(float, (row["xmin"], row["ymin"], row["xmax"], row["ymax"]))

                x_center = ((xmin + xmax) / 2) / image_width
                y_center = ((ymin + ymax) / 2) / image_height
                box_width = (xmax - xmin) / image_width
                box_height = (ymax - ymin) / image_height
                class_id = class_to_id[str(row["class"])]

                label_lines.append(
                    f"{class_id} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}"
                )

            label_path = split_dir / f"{Path(str(image_name)).stem}.txt"
            label_path.write_text("\n".join(label_lines) + "\n", encoding="utf-8")

        print(f"✅ {split} bölümü YOLO formatına dönüştürüldü.")

    print(f"✅ Tespit edilen sınıflar: {class_names}")
    return class_names


if DATASET_READY:
    CLASS_NAMES = convert_csv_to_yolo_labels()
else:
    CLASS_NAMES = []

## 4. YOLO Veri Yapılandırmasını Oluşturma

YOLOv8 eğitimi için `data/dental_data.yaml` dosyası kullanılır. Bu dosya eğitim, doğrulama ve test yollarını; sınıf sayısını ve sınıf isimlerini içerir.

In [ ]:
def write_yolo_data_yaml(class_names: list[str], yaml_path: Path = DATA_YAML_PATH) -> Path:
    """YOLOv8 eğitiminde kullanılacak data YAML dosyasını oluşturur."""
    if not class_names:
        raise ValueError("Sınıf listesi boş. Önce CSV -> YOLO dönüşümünü tamamlayın.")

    yaml_data = {
        "train": str((DATA_DIR / "train").resolve()),
        "val": str((DATA_DIR / "valid").resolve()),
        "test": str((DATA_DIR / "test").resolve()),
        "nc": len(class_names),
        "names": class_names,
    }

    yaml_path.parent.mkdir(parents=True, exist_ok=True)
    yaml_path.write_text(
        yaml.safe_dump(yaml_data, allow_unicode=True, sort_keys=False),
        encoding="utf-8",
    )
    return yaml_path


if DATASET_READY:
    yaml_path = write_yolo_data_yaml(CLASS_NAMES)
    print(f"✅ YOLO yapılandırması oluşturuldu: {yaml_path}")
else:
    print("⚠️ Veri seti hazır olmadığı için YAML oluşturma adımı atlandı.")

## 5. YOLOv8 Modelini Eğitme

Bu projede temel model olarak `yolov8n.pt` kullanılmıştır. `YOLOv8n`, hızlı deneme ve portföy prototipi için hafif bir başlangıç modelidir.

Eğitim tamamlandıktan sonra en iyi ağırlık dosyası yerel olarak şu konuma kaydedilir:

```text
runs/detect/dental_yolov8n/weights/best.pt
```

Portföy demosunda kullanılan ağırlık dosyasını güncellemek için bu dosya `models/best.pt` konumuna da kopyalanır.

In [ ]:
if DATASET_READY:
    model = YOLO("yolov8n.pt")

    train_results = model.train(
        data=str(DATA_YAML_PATH),
        epochs=10,
        imgsz=512,
        batch=16,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
    )

    if TRAINED_WEIGHTS_PATH.exists():
        PORTFOLIO_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(TRAINED_WEIGHTS_PATH, PORTFOLIO_MODEL_PATH)
        print(f"✅ En iyi model ağırlığı güncellendi: {PORTFOLIO_MODEL_PATH}")
    else:
        print(f"⚠️ Beklenen ağırlık dosyası bulunamadı: {TRAINED_WEIGHTS_PATH}")
else:
    print("⚠️ Ham veri seti hazır olmadığı için eğitim hücresi çalıştırılmadı.")

## 6. Eğitilen Modeli Test Görseli Üzerinde Deneme

Bu bölüm, eğitim tamamlandıktan sonra test klasöründen ilk uygun görseli seçer ve modelin ürettiği işaretli çıktıyı orijinal röntgenle yan yana gösterir.

In [ ]:
def find_first_test_image() -> Path | None:
    test_dir = DATA_DIR / "test"
    if not test_dir.exists():
        return None

    for image_path in sorted(test_dir.iterdir()):
        if image_path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS:
            return image_path
    return None


def show_prediction(image_path: Path, model_path: Path = PORTFOLIO_MODEL_PATH, confidence: float = 0.25) -> None:
    if not model_path.exists():
        raise FileNotFoundError(f"Model ağırlığı bulunamadı: {model_path}")

    model = YOLO(str(model_path))
    results = model(str(image_path), conf=confidence)

    original_bgr = cv2.imread(str(image_path))
    if original_bgr is None:
        raise FileNotFoundError(f"Görsel okunamadı: {image_path}")

    original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
    predicted_rgb = cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB)

    print("=" * 60)
    print("🦷 DENTAL AI TESPİT RAPORU")
    print(f"Dosya: {image_path.name}")
    print("-" * 60)

    if len(results[0].boxes) == 0:
        print("Herhangi bir bulgu tespit edilmedi.")
    else:
        for box in results[0].boxes:
            class_id = int(box.cls[0])
            confidence_score = float(box.conf[0])
            class_name = model.names[class_id]
            print(f"📌 {class_name} | güven: %{confidence_score * 100:.2f}")

    print("=" * 60)

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(original_rgb)
    axes[0].set_title("Orijinal Röntgen", fontsize=14, fontweight="bold")
    axes[0].axis("off")

    axes[1].imshow(predicted_rgb)
    axes[1].set_title("Model Tahmini", fontsize=14, fontweight="bold")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()


sample_image = find_first_test_image()
if sample_image is not None and PORTFOLIO_MODEL_PATH.exists():
    show_prediction(sample_image)
else:
    print("⚠️ Test görseli veya model ağırlığı bulunamadı. Eğitim tamamlandıktan sonra bu hücreyi çalıştırın.")

## 7. Eğitim Performansını Görselleştirme

YOLOv8 eğitimi tamamlandığında `results.csv` dosyası oluşur. Bu dosya üzerinden loss ve mAP metrikleri görselleştirilebilir.

In [ ]:
results_csv_path = RUNS_DIR / RUN_NAME / "results.csv"

if results_csv_path.exists():
    df = pd.read_csv(results_csv_path)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(df["epoch"], df["train/cls_loss"], label="Eğitim sınıflandırma kaybı")
    axes[0].plot(df["epoch"], df["val/cls_loss"], label="Doğrulama sınıflandırma kaybı")
    axes[0].set_title("Sınıflandırma Loss Karşılaştırması")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, linestyle="--", alpha=0.6)

    axes[1].plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP@0.50")
    axes[1].set_title("Model Tespit Başarısı")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("mAP")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()
    plt.show()
else:
    print(f"⚠️ Eğitim sonuç dosyası bulunamadı: {results_csv_path}")

## 8. Notlar

- `runs/` klasörü `.gitignore` kapsamındadır; eğitim çıktıları GitHub'a eklenmez.
- Portföy demosunda kullanılan küçük model ağırlığı `models/best.pt` olarak repoda tutulur.
- Ham veri seti boyut nedeniyle repoya eklenmemiştir.
- Görsel demo ve LinkedIn ekran görüntüsü için `02_model_demo.ipynb` kullanılması önerilir.